# post graduate

In [ ]:
# @title main Masters script

import requests
from bs4 import BeautifulSoup
import os
import json
import time
import concurrent.futures
from datetime import datetime

# ==========================================
#        MASTER CONFIGURATION
# ==========================================

LINKS_FILE = 'result_tcc_links_json.txt'
COLLEGE_CODE_FILE = 'college-codes.txt'
AJAX_URL = "https://durg.ucanapply.com/get-result-details"

MAX_WORKERS = 15               # Number of parallel requests
MAX_SERIAL_PER_COLLEGE = 80    # Max class size to check
EARLY_STOP_THRESHOLD = 5       # Skip college if 001-005 don't exist
CONSECUTIVE_FAIL_LIMIT = 8     # Stop if 8 consecutive students missing

BASE_FOLDER = "Durg_University_Master_Archive"

# ==========================================
#      SUBJECT & COURSE CODE MAPPING
# ==========================================

SUBJECTS = {
    # M.Sc Subjects
    "MSc_Chemistry":      {"old": "093", "new": "77", "kw": ["Chemistry"]},
    "MSc_Botany":         {"old": "073", "new": "62", "kw": ["Botany"]},
    "MSc_Zoology":        {"old": "077", "new": "63", "kw": ["Zoology"]},
    "MSc_Mathematics":    {"old": "081", "new": "64", "kw": ["Mathematics"]},
    "MSc_Microbiology":   {"old": "101", "new": "82", "kw": ["Microbiology"]},
    "MSc_Biotechnology":  {"old": "089", "new": "68", "kw": ["Bio Technology", "Biotechnology"]},
    "MSc_Physics":        {"old": "085", "new": "67", "kw": ["Physics"]},
    "MSc_Comp_Science":   {"old": "097", "new": "81", "kw": ["Computer Science", "M.Sc.C.S"]},
    "MSc_HomeSci_Human":  {"old": None,  "new": "80", "kw": ["Human Development"]},
    "MSc_HomeSci_Food":   {"old": None,  "new": "85", "kw": ["Food Science"]},
    "MSc_HomeSci_Textile":{"old": None,  "new": "83", "kw": ["Textile and Clothing"]},

    # M.A Subjects
    "MA_English":         {"old": "041", "new": "55", "kw": ["Arts in English"]},
    "MA_Hindi":           {"old": "037", "new": "75", "kw": ["Arts in Hindi"]},
    "MA_History":         {"old": "065", "new": "60", "kw": ["Arts in History"]},
    "MA_Political_Sci":   {"old": "057", "new": "59", "kw": ["Political Science"]},
    "MA_Economics":       {"old": "049", "new": "57", "kw": ["Arts in Economics"]},
    "MA_Sociology":       {"old": "045", "new": "56", "kw": ["Arts in Sociology"]},
    "MA_Geography":       {"old": "053", "new": "58", "kw": ["Arts in Geography"]},
    "MA_Psychology":      {"old": "069", "new": "61", "kw": ["Arts in Psychology"]},
    "MA_Home_Science":    {"old": "061", "new": "80", "kw": ["Arts in Home Science"]}
}

# ==========================================
#     BATCH TIMELINES & ROLL ALGORITHMS
# ==========================================

BATCHES = [
    {
        "id": "B1_2018_19", "code_type": "old",
        "roll_algo": lambda coll, c, i: f"{coll}18{c}{i:03d}", # College first
        "timeline": [
            {"sem": "First Semester",  "yr": 2019},
            {"sem": "Second Semester", "yr": 2019},
            {"sem": "Third Semester",  "yr": 2020},
            {"sem": "Fourth Semester", "yr": 2020}
        ]
    },
    {
        "id": "B2_2019_20", "code_type": "old",
        "roll_algo": lambda coll, c, i: f"19{coll}{c}{i:03d}",
        "timeline": [
            {"sem": "First Semester",  "yr": 2020},
            {"sem": "Second Semester", "yr": 2020},
            {"sem": "Third Semester",  "yr": 2021},
            {"sem": "Fourth Semester", "yr": 2021}
        ]
    },
    {
        "id": "B3_2020_21", "code_type": "old",
        "roll_algo": lambda coll, c, i: f"20{coll}{c}{i:03d}",
        "timeline": [
            {"sem": "First Semester",  "yr": 2021},
            {"sem": "Second Semester", "yr": 2021},
            {"sem": "Third Semester",  "yr": 2022},
            {"sem": "Fourth Semester", "yr": 2022}
        ]
    },
    {
        "id": "B4_2021_22", "code_type": "old",
        "roll_algo": lambda coll, c, i: f"21{coll}{c}{i:03d}",
        "timeline": [
            {"sem": "First Semester",  "yr": 2022},
            {"sem": "Second Semester", "yr": 2022},
            {"sem": "Third Semester",  "yr": 2023},
            {"sem": "Fourth Semester", "yr": 2023}
        ]
    },
    {
        "id": "B5_2022_23", "code_type": "old",
        "roll_algo": lambda coll, c, i: f"23{coll}{c}{i:04d}", # 4-digit serial
        "timeline": [
            {"sem": "First Semester",  "yr": 2023},
            {"sem": "Second Semester", "yr": 2023},
            {"sem": "Third Semester",  "yr": 2024},
            {"sem": "Fourth Semester", "yr": 2024}
        ]
    },
    {
        "id": "B6_2023_24", "code_type": "new",
        "roll_algo": lambda coll, c, i: f"24{coll}{c}{i:03d}",
        "timeline": [
            {"sem": "First Semester",  "yr": 2024},
            {"sem": "Second Semester", "yr": 2024},
            {"sem": "Third Semester",  "yr": 2025},
            {"sem": "Fourth Semester", "yr": 2025} # Future
        ]
    },
    {
        "id": "B7_2024_25", "code_type": "new",
        "roll_algo": lambda coll, c, i: f"25{coll}{c}{i:03d}",
        "timeline": [
            {"sem": "First Semester",  "yr": 2025},
            {"sem": "Second Semester", "yr": 2025}, # Future
            {"sem": "Third Semester",  "yr": 2026}, # Future
            {"sem": "Fourth Semester", "yr": 2026} # Future
        ]
    }
]

# ==========================================
#           CORE FUNCTIONS
# ==========================================

def load_json_data():
    if not os.path.exists(LINKS_FILE):
        print(f"CRITICAL: {LINKS_FILE} not found.")
        return []
    with open(LINKS_FILE, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_college_codes():
    codes = []
    if not os.path.exists(COLLEGE_CODE_FILE):
        return []
    with open(COLLEGE_CODE_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if parts[0].isdigit():
                codes.append(parts[0])
    return codes

def find_exact_link(json_data, subj_kws, sem_str, tgt_year):
    """
    Intelligently searches JSON for the correct Exam Link based on:
    1. Subject Keywords
    2. Semester String ("First Semester")
    3. Declaration Year (To separate different batches)
    4. Ensures it's the "REGULAR / PRIVATE" list, not ATKT/SUPPLY.
    """
    for item in json_data:
        title = item['title'].lower()

        # 1. Subject Match
        if not any(kw.lower() in title for kw in subj_kws):
            continue

        # 2. Semester Match
        if sem_str.lower() not in title:
            continue

        # 3. STRICT REGULAR MATCH
        if "regular" not in title:
            continue

        # 4. Date Match
        try:
            date_obj = datetime.strptime(item['date'], "%d/%m/%Y")
            if date_obj.year == tgt_year:
                return item
        except:
            pass

    return None

def get_session_tokens(url):
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36',
        'Referer': url
    })

    try:
        response = session.get(url, timeout=10)
        if response.status_code != 200: return None, None, None

        soup = BeautifulSoup(response.text, 'html.parser')
        csrf_input = soup.find('input', {'name': '_token'})
        if not csrf_input: return None, None, None

        payload = {
            'COURSECD': soup.find('input', {'name': 'COURSECD'})['value'],
            'SEMCODE': soup.find('input', {'name': 'SEMCODE'})['value'],
            'RESULTTYPE': soup.find('input', {'name': 'RESULTTYPE'})['value'],
            'session': soup.find('input', {'name': 'session'})['value'],
            'tcc': soup.find('input', {'name': 'tcc'})['value'],
            'p1': '', 'all': ''
        }
        return session.cookies.get_dict(), payload, csrf_input['value']
    except:
        return None, None, None

def fetch_worker(args):
    roll, url, template, token, cookies = args
    headers = {
        'X-Requested-With': 'XMLHttpRequest',
        'X-CSRF-TOKEN': token,
        'Origin': 'https://durg.ucanapply.com',
        'Referer': url,
        'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8'
    }
    data = template.copy()
    data['EXAMROLLNUMBER'] = roll
    data['_token'] = token

    try:
        res = requests.post(AJAX_URL, data=data, headers=headers, cookies=cookies, timeout=5)
        if res.status_code == 200:
            res_json = res.json()
            if 'html' in res_json and len(res_json['html']) > 100: # Verify it's a real result
                return (roll, res_json['html'])
    except:
        pass
    return (roll, None)

# ==========================================
#           PROCESSING ENGINE
# ==========================================

def main():
    links_data = load_json_data()
    colleges = load_college_codes()

    if not links_data or not colleges:
        print("Required files missing. Aborting.")
        return

    print(f"🚀 Starting ULTIMATE MA/MSc Scraper | Batches B1 to B7")
    print("=" * 75)

    for batch in BATCHES:

        for subj_name, subj_data in SUBJECTS.items():

            course_code = subj_data[batch['code_type']]
            if not course_code:
                continue

            for sem_info in batch['timeline']:

                # Strict Link Finder (Will ONLY grab "REGULAR / PRIVATE")
                link_item = find_exact_link(
                    links_data,
                    subj_data['kw'],
                    sem_info['sem'],
                    sem_info['yr']
                )

                if not link_item:
                    continue

                folder_path = os.path.join(BASE_FOLDER, batch['id'], subj_name, sem_info['sem'].replace(" ", "_"))
                os.makedirs(folder_path, exist_ok=True)

                print(f"\n📁 {batch['id']} | 📘 {subj_name} | 🗓️ {sem_info['sem']}")
                print(f"🔗 Target Link: {link_item['title']} (Dec: {link_item['date']})")

                cookies, template, token = get_session_tokens(link_item['url'])
                if not token:
                    print("❌ Error: Link expired or session failed.")
                    continue

                total_found = 0

                for college in colleges:
                    college_active = False
                    consecutive_fails = 0

                    tasks = []
                    for i in range(1, MAX_SERIAL_PER_COLLEGE + 1):
                        roll_number = batch['roll_algo'](college, course_code, i)
                        tasks.append((roll_number, link_item['url'], template, token, cookies))

                    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                        for roll, html_content in executor.map(fetch_worker, tasks):

                            print(f"\r🏢 Col {college} | 🔍 Checking | ✅ Saved: {total_found} ", end="", flush=True)

                            if html_content:
                                college_active = True
                                consecutive_fails = 0
                                total_found += 1

                                filepath = os.path.join(folder_path, f"{roll}.html")
                                with open(filepath, "w", encoding="utf-8") as f:
                                    f.write(html_content)
                            else:
                                consecutive_fails += 1

                            # Smart Skip Logic
                            serial_int = int(roll[-4:]) if batch['id'] == 'B5_2022_23' else int(roll[-3:])

                            if not college_active and serial_int >= EARLY_STOP_THRESHOLD:
                                executor.shutdown(wait=False, cancel_futures=True)
                                break

                            if college_active and consecutive_fails >= CONSECUTIVE_FAIL_LIMIT:
                                executor.shutdown(wait=False, cancel_futures=True)
                                break

                if total_found > 0:
                    print(f"\n✅ Finished {subj_name}. Total Saved: {total_found}")
                else:
                    print(f"\n⚠️ Finished. No results found.")

                time.sleep(1) # Polite cooldown between subjects

    print(f"\n\n{'=' * 75}")
    print("🎉🎉 THE MASTER SCRAPE IS COMPLETELY FINISHED 🎉🎉")

if __name__ == "__main__":
    main()

In [ ]:
# @title other master script
import requests
from bs4 import BeautifulSoup
import os
import json
import time
import concurrent.futures
from datetime import datetime

# ==========================================
#        MASTER CONFIGURATION
# ==========================================

LINKS_FILE = 'result_tcc_links_json.txt'
COLLEGE_CODE_FILE = 'college-codes.txt'
AJAX_URL = "https://durg.ucanapply.com/get-result-details"

MAX_WORKERS = 15
MAX_SERIAL_PER_COLLEGE = 80
EARLY_STOP_THRESHOLD = 5
CONSECUTIVE_FAIL_LIMIT = 8

BASE_FOLDER = "Professional_PG_Archive"

# ==========================================
#      ONLY PROFESSIONAL PG SUBJECTS
# ==========================================

SUBJECTS = {
    "MCom":               {"old": "117", "new": "76", "kw": ["Master of Commerce", "M.Com"]},
    "MEd":                {"old": "129", "new": "79", "kw": ["Master of Education", "M.Ed"]},
    "MLib":               {"old": "121", "new": "69", "kw": ["Library", "M.Lib"]},
    "LLM":                {"old": "118", "new": "70", "kw": ["Master of Laws", "LL.M", "LLM"]},
    "MSW":                {"old": "125", "new": "74", "kw": ["Social Work", "M.S.W"]}
}

# ==========================================
#     BATCH TIMELINES & ROLL ALGORITHMS
# ==========================================

BATCHES = [
    {
        "id": "B1_2018_19", "code_type": "old",
        "roll_algo": lambda coll, c, i, s: f"{coll}18{c}{i:03d}", # College first logic
        "timeline": [{"sem": "First Semester", "yr": 2019}, {"sem": "Second Semester", "yr": 2019}, {"sem": "Third Semester", "yr": 2020}, {"sem": "Fourth Semester", "yr": 2020}]
    },
    {
        "id": "B2_2019_20", "code_type": "old",
        "roll_algo": lambda coll, c, i, s: f"19{coll}{c}{i:03d}",
        "timeline": [{"sem": "First Semester", "yr": 2020}, {"sem": "Second Semester", "yr": 2020}, {"sem": "Third Semester", "yr": 2021}, {"sem": "Fourth Semester", "yr": 2021}]
    },
    {
        "id": "B3_2020_21", "code_type": "old",
        "roll_algo": lambda coll, c, i, s: f"20{coll}{c}{i:03d}",
        "timeline": [{"sem": "First Semester", "yr": 2021}, {"sem": "Second Semester", "yr": 2021}, {"sem": "Third Semester", "yr": 2022}, {"sem": "Fourth Semester", "yr": 2022}]
    },
    {
        "id": "B4_2021_22", "code_type": "old",
        "roll_algo": lambda coll, c, i, s: f"21{coll}{c}{i:03d}",
        "timeline": [{"sem": "First Semester", "yr": 2022}, {"sem": "Second Semester", "yr": 2022}, {"sem": "Third Semester", "yr": 2023}, {"sem": "Fourth Semester", "yr": 2023}]
    },
    {
        "id": "B5_2022_23", "code_type": "old",
        # NOTE: B5 uses variable serial length (s) passed from main loop
        "roll_algo": lambda coll, c, i, s: f"23{coll}{c}{i:0{s}d}",
        "timeline": [{"sem": "First Semester", "yr": 2023}, {"sem": "Second Semester", "yr": 2023}, {"sem": "Third Semester", "yr": 2024}, {"sem": "Fourth Semester", "yr": 2024}]
    },
    {
        "id": "B6_2023_24", "code_type": "new",
        # NOTE: B6 usually standardizes back to 3 digits for new codes, but check mapping if needed.
        # Based on your M.Com example: 2410476003 -> 3 digits.
        "roll_algo": lambda coll, c, i, s: f"24{coll}{c}{i:03d}",
        "timeline": [{"sem": "First Semester", "yr": 2024}, {"sem": "Second Semester", "yr": 2024}, {"sem": "Third Semester", "yr": 2025}, {"sem": "Fourth Semester", "yr": 2025}]
    },
    {
        "id": "B7_2024_25", "code_type": "new",
        "roll_algo": lambda coll, c, i, s: f"25{coll}{c}{i:03d}",
        "timeline": [{"sem": "First Semester", "yr": 2025}, {"sem": "Second Semester", "yr": 2025}, {"sem": "Third Semester", "yr": 2026}, {"sem": "Fourth Semester", "yr": 2026}]
    }
]

# ==========================================
#           CORE FUNCTIONS
# ==========================================

def load_json_data():
    if not os.path.exists(LINKS_FILE):
        return []
    with open(LINKS_FILE, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_college_codes():
    codes = []
    if not os.path.exists(COLLEGE_CODE_FILE):
        return []
    with open(COLLEGE_CODE_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if parts[0].isdigit(): codes.append(parts[0])
    return codes

def find_exact_link(json_data, subj_kws, sem_str, tgt_year):
    for item in json_data:
        title = item['title'].lower()
        if not any(kw.lower() in title for kw in subj_kws): continue
        if sem_str.lower() not in title: continue
        if "regular" not in title: continue # STRICT REGULAR FILTER
        try:
            date_obj = datetime.strptime(item['date'], "%d/%m/%Y")
            if date_obj.year == tgt_year: return item
        except: pass
    return None

def get_session_tokens(url):
    session = requests.Session()
    session.headers.update({'User-Agent': 'Mozilla/5.0', 'Referer': url})
    try:
        response = session.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        csrf_token = soup.find('input', {'name': '_token'})['value']
        payload = {
            'COURSECD': soup.find('input', {'name': 'COURSECD'})['value'],
            'SEMCODE': soup.find('input', {'name': 'SEMCODE'})['value'],
            'RESULTTYPE': soup.find('input', {'name': 'RESULTTYPE'})['value'],
            'session': soup.find('input', {'name': 'session'})['value'],
            'tcc': soup.find('input', {'name': 'tcc'})['value'],
            'p1': '', 'all': ''
        }
        return session.cookies.get_dict(), payload, csrf_token
    except: return None, None, None

def fetch_worker(args):
    roll, url, template, token, cookies = args
    headers = {'X-Requested-With': 'XMLHttpRequest', 'X-CSRF-TOKEN': token, 'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8'}
    data = template.copy()
    data['EXAMROLLNUMBER'] = roll
    data['_token'] = token
    try:
        res = requests.post(AJAX_URL, data=data, headers=headers, cookies=cookies, timeout=5)
        if res.status_code == 200:
            rj = res.json()
            if 'html' in rj and len(rj['html']) > 100: return (roll, rj['html'])
    except: pass
    return (roll, None)

# ==========================================
#           PROCESSING ENGINE
# ==========================================

def main():
    links_data = load_json_data()
    colleges = load_college_codes()

    print(f"🚀 Starting PROFESSIONAL PG Scraper (M.Com, M.Ed, LLM, MSW, M.Lib)")
    print(f"📁 Saving to: {BASE_FOLDER}")

    for batch in BATCHES:
        for subj_name, subj_data in SUBJECTS.items():
            course_code = subj_data[batch['code_type']]
            if not course_code: continue

            # Special 4-digit serial logic for B5 batch for these specific subjects
            serial_len = 4 if batch['id'] == "B5_2022_23" else 3

            for sem_info in batch['timeline']:
                link_item = find_exact_link(links_data, subj_data['kw'], sem_info['sem'], sem_info['yr'])
                if not link_item: continue

                folder_path = os.path.join(BASE_FOLDER, batch['id'], subj_name, sem_info['sem'].replace(" ", "_"))
                os.makedirs(folder_path, exist_ok=True)

                print(f"\n📁 {batch['id']} | {subj_name} | {sem_info['sem']}")
                print(f"🔗 {link_item['title']} ({link_item['date']})")

                cookies, template, token = get_session_tokens(link_item['url'])
                if not token: continue

                total_found = 0
                for college in colleges:
                    college_active, consecutive_fails = False, 0
                    tasks = []

                    for i in range(1, MAX_SERIAL_PER_COLLEGE + 1):
                        # Generate Roll Number
                        roll = batch['roll_algo'](college, course_code, i, serial_len)
                        tasks.append((roll, link_item['url'], template, token, cookies))

                    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                        for roll, html in executor.map(fetch_worker, tasks):
                            print(f"\r🏢 Col {college} | ✅ Saved: {total_found} ", end="", flush=True)

                            if html:
                                college_active, consecutive_fails = True, 0
                                total_found += 1
                                with open(os.path.join(folder_path, f"{roll}.html"), "w", encoding="utf-8") as f: f.write(html)
                            else: consecutive_fails += 1

                            # Smart Skip Logic
                            # Use regex or simple slicing to get the serial part for threshold check
                            idx = int(roll[-serial_len:])
                            if not college_active and idx >= EARLY_STOP_THRESHOLD:
                                executor.shutdown(wait=False, cancel_futures=True); break
                            if college_active and consecutive_fails >= CONSECUTIVE_FAIL_LIMIT:
                                executor.shutdown(wait=False, cancel_futures=True); break

                print(f"\n✅ Done. Total: {total_found}")

if __name__ == "__main__":
    main()

# under graduate

In [ ]:
# @title bsc final 2023 and 2022
import requests
from bs4 import BeautifulSoup
import concurrent.futures
import time
import os

# ==========================================
# CONFIG (2022 STRUCTURE)
# ==========================================

# New 2022 Result Link
RESULT_PAGE_URL = "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6ImJvRGFrbHJHaEFBOGM4VDUrNWRLQmc9PSIsInZhbHVlIjoiM1pJdHZsVmFqWUtMTExJY0Y4VXFYckZONXl0S1dOMmVxaXU4cmFGMmxqbz0iLCJtYWMiOiIxMjM3ZjFmMmU1MWJhNzMyMTVjYTY0NGMyM2MzY2RiMmZmM2FhOWMyZjg0ZWFmMjNhMzk3NDAzOTI0NmY3ZWY4IiwidGFnIjoiIn0="
AJAX_URL = "https://durg.ucanapply.com/get-result-details"

COLLEGE_FILE = "college-codes.txt"

# 2022 Roll Structure: 23 (Year) + CollegeCode + 007 (Course) + Serial
YEAR_CONST = "3"
COURSE_CONST = "008" # 007 for 22

MAX_WORKERS = 200
REQUESTS_PER_SECOND = 200
SERIAL_START = 1
SERIAL_END = 300

SAVE_FOLDER = "UG_BSC_FINAL_2023"
os.makedirs(SAVE_FOLDER, exist_ok=True)

# ==========================================
# LOAD COLLEGE CODES
# ==========================================

def load_college_codes():
    codes = []
    if not os.path.exists(COLLEGE_FILE):
        print(f"❌ Error: {COLLEGE_FILE} not found!")
        return []
    with open(COLLEGE_FILE, "r", encoding="utf-8") as f:
        lines = f.readlines()[1:]
        for line in lines:
            parts = line.strip().split("\t")
            if parts and parts[0].isdigit():
                codes.append(parts[0])
    return codes

# ==========================================
# TOKEN FETCH
# ==========================================

def get_session_tokens():
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": RESULT_PAGE_URL
    })

    try:
        r = session.get(RESULT_PAGE_URL, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")

        csrf_token = soup.find("input", {"name": "_token"})["value"]

        payload_template = {
            "COURSECD": soup.find("input", {"name": "COURSECD"})["value"],
            "SEMCODE": soup.find("input", {"name": "SEMCODE"})["value"],
            "RESULTTYPE": soup.find("input", {"name": "RESULTTYPE"})["value"],
            "session": soup.find("input", {"name": "session"})["value"],
            "tcc": soup.find("input", {"name": "tcc"})["value"],
            "p1": "",
            "all": ""
        }
        return session.cookies.get_dict(), payload_template, csrf_token
    except Exception as e:
        print(f"❌ Failed to fetch initial tokens: {e}")
        exit()

# ==========================================
# WORKER
# ==========================================

def fetch_roll(args):
    roll, template, token, cookies = args

    headers = {
        "X-Requested-With": "XMLHttpRequest",
        "X-CSRF-TOKEN": token,
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "User-Agent": "Mozilla/5.0"
    }

    data = template.copy()
    data["EXAMROLLNUMBER"] = roll
    data["_token"] = token

    try:
        res = requests.post(AJAX_URL, data=data, headers=headers, cookies=cookies, timeout=10)
        if res.status_code == 200:
            rj = res.json()
            if "html" in rj and len(rj["html"]) > 100:
                with open(os.path.join(SAVE_FOLDER, f"{roll}.html"), "w", encoding="utf-8") as f:
                    f.write(rj["html"])
                return "SAVED"
    except:
        pass

    return "FAILED"

# ==========================================
# MAIN ENGINE
# ==========================================

def main():
    print("🚀 UG BSC FINAL 2022 MULTI-COLLEGE SCRAPER")
    print(f"📁 Target Folder: {SAVE_FOLDER}")

    # Resume scan
    existing_files = set(f.replace(".html", "") for f in os.listdir(SAVE_FOLDER) if f.endswith(".html"))
    if existing_files:
        print(f"🔎 Found {len(existing_files)} existing files. Skipping...")

    colleges = load_college_codes()
    if not colleges: return

    cookies, template, token = get_session_tokens()
    total_newly_saved = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

        for college in colleges:
            print(f"\n🏫 Processing College: {college}")

            consecutive_failures = 0
            # Check if we already have files for this college
            started = any(roll.startswith(f"{YEAR_CONST}{college}") for roll in existing_files)

            for batch_start in range(SERIAL_START, SERIAL_END + 1, REQUESTS_PER_SECOND):

                batch_end = min(batch_start + REQUESTS_PER_SECOND - 1, SERIAL_END)
                serial_range = range(batch_start, batch_end + 1)

                tasks = []
                for serial in serial_range:
                    # 2022 Format: 23 + College + 007 + Serial
                    roll = f"{YEAR_CONST}{college}{COURSE_CONST}{serial:04d}"

                    if roll in existing_files:
                        continue

                    tasks.append((roll, template, token, cookies))

                if not tasks:
                    continue

                start_time = time.time()
                results = list(executor.map(fetch_roll, tasks))

                batch_stop_triggered = False
                for status in results:
                    if status == "SAVED":
                        total_newly_saved += 1
                        consecutive_failures = 0
                        started = True
                        print(f"✅ New Saved: {total_newly_saved} | Current: {college}", end="\r")
                    else:
                        if started:
                            consecutive_failures += 1
                            if consecutive_failures >= 5:
                                print(f"\n🛑 Stopping College {college} (5 consecutive empty results)")
                                batch_stop_triggered = True
                                break

                if batch_stop_triggered:
                    break

                # Maintain rate limit
                elapsed = time.time() - start_time
                if elapsed < 1:
                    time.sleep(1 - elapsed)

    print(f"\n\n🎉 Task Finished.")
    print(f"📊 Newly Downloaded: {total_newly_saved}")
    print(f"📂 Total records in folder: {len(os.listdir(SAVE_FOLDER))}")

if __name__ == "__main__":
    main()

In [ ]:
# @title bsc final 2025
import requests
from bs4 import BeautifulSoup
import concurrent.futures
import time
import os

# ==========================================
# CONFIG
# ==========================================

RESULT_PAGE_URL = "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6IlgzVlh4QldpRy9RMmZPYjR5VkVOQ2c9PSIsInZhbHVlIjoiWXV6WEdQaDlhOWpmMHY0NFdUL0xadEhaRHZBQitjTWFXM0s4UTJHTEVxUT0iLCJtYWMiOiI1NDRjM2M1NDk0YWM5NWYyZjk4NzYxZmY2Yjg0ODAwNGU0ODRiMWMyOTk4NjUxZWQ4NDIzNzgwMTkzMTVjYzc4IiwidGFnIjoiIn0="
AJAX_URL = "https://durg.ucanapply.com/get-result-details"

COLLEGE_FILE = "college-code.txt"

MAX_WORKERS = 300
REQUESTS_PER_SECOND = 300
SERIAL_START = 1
SERIAL_END = 9999   # Change if needed

SAVE_FOLDER = "UG_BSC_FINAL_2025"
os.makedirs(SAVE_FOLDER, exist_ok=True)

# ==========================================
# LOAD COLLEGE CODES
# ==========================================

def load_college_codes():
    codes = []
    if not os.path.exists(COLLEGE_FILE):
        print(f"❌ Error: {COLLEGE_FILE} not found!")
        return []
    with open(COLLEGE_FILE, "r", encoding="utf-8") as f:
        lines = f.readlines()[1:]  # skip header
        for line in lines:
            parts = line.strip().split("\t")
            if parts and parts[0].isdigit():
                codes.append(parts[0])
    return codes

# ==========================================
# TOKEN FETCH
# ==========================================

def get_session_tokens():
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": RESULT_PAGE_URL
    })

    try:
        r = session.get(RESULT_PAGE_URL, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")

        csrf_token = soup.find("input", {"name": "_token"})["value"]

        payload_template = {
            "COURSECD": soup.find("input", {"name": "COURSECD"})["value"],
            "SEMCODE": soup.find("input", {"name": "SEMCODE"})["value"],
            "RESULTTYPE": soup.find("input", {"name": "RESULTTYPE"})["value"],
            "session": soup.find("input", {"name": "session"})["value"],
            "tcc": soup.find("input", {"name": "tcc"})["value"],
            "p1": "",
            "all": ""
        }
        return session.cookies.get_dict(), payload_template, csrf_token
    except Exception as e:
        print(f"❌ Failed to fetch initial tokens: {e}")
        exit()

# ==========================================
# WORKER
# ==========================================

def fetch_roll(args):
    roll, template, token, cookies = args

    headers = {
        "X-Requested-With": "XMLHttpRequest",
        "X-CSRF-TOKEN": token,
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "User-Agent": "Mozilla/5.0"
    }

    data = template.copy()
    data["EXAMROLLNUMBER"] = roll
    data["_token"] = token

    try:
        res = requests.post(AJAX_URL, data=data, headers=headers, cookies=cookies, timeout=10)
        if res.status_code == 200:
            rj = res.json()
            if "html" in rj and len(rj["html"]) > 100:
                with open(os.path.join(SAVE_FOLDER, f"{roll}.html"), "w", encoding="utf-8") as f:
                    f.write(rj["html"])
                return "SAVED"
    except:
        pass

    return "FAILED"

# ==========================================
# MAIN ENGINE
# ==========================================

def main():
    print("🚀 UG BSC FINAL 25 MULTI-COLLEGE SCRAPER (With Auto-Resume)")
    print(f"📁 Saving to: {SAVE_FOLDER}")

    # Check existing files to avoid re-downloading
    existing_files = set(f.replace(".html", "") for f in os.listdir(SAVE_FOLDER) if f.endswith(".html"))
    print(f"🔎 Found {len(existing_files)} existing records. These will be skipped.")

    colleges = load_college_codes()
    if not colleges: return

    cookies, template, token = get_session_tokens()
    total_newly_saved = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

        for college in colleges:
            print(f"\n🏫 Processing College: {college}")

            consecutive_failures = 0
            # If we already have some files for this college, we assume we 'started'
            started = any(roll.startswith(f"5{college}") for roll in existing_files)

            for batch_start in range(SERIAL_START, SERIAL_END + 1, REQUESTS_PER_SECOND):

                batch_end = min(batch_start + REQUESTS_PER_SECOND - 1, SERIAL_END)
                serial_range = range(batch_start, batch_end + 1)

                tasks = []
                for serial in serial_range:
                    roll = f"5{college}{serial:04d}"

                    # --- RESUME LOGIC ---
                    if roll in existing_files:
                        continue # Skip if already on disk

                    tasks.append((roll, template, token, cookies))

                if not tasks:
                    # If whole batch was skipped, just move to next batch
                    continue

                start_time = time.time()
                results = list(executor.map(fetch_roll, tasks))

                batch_stop_triggered = False
                for status in results:
                    if status == "SAVED":
                        total_newly_saved += 1
                        consecutive_failures = 0
                        started = True
                        print(f"✅ New Saved: {total_newly_saved} | Current: {college}", end="\r")
                    else:
                        if started:
                            consecutive_failures += 1
                            if consecutive_failures >= 3:
                                print(f"\n🛑 Stopping College {college} (3 consecutive empty results)")
                                batch_stop_triggered = True
                                break

                if batch_stop_triggered:
                    break

                # Rate limiting throttle
                elapsed = time.time() - start_time
                if elapsed < 1:
                    time.sleep(1 - elapsed)

    print(f"\n\n🎉 Task Finished.")
    print(f"📊 Newly Downloaded: {total_newly_saved}")
    print(f"📂 Total files in folder: {len(os.listdir(SAVE_FOLDER))}")

if __name__ == "__main__":
    main()

In [ ]:
# @title bsc 2020
import requests
from bs4 import BeautifulSoup
import concurrent.futures
import time
import os

# ==========================================
# CONFIG (2020 STRUCTURE)
# ==========================================

# New 2020 Result Link
RESULT_PAGE_URL = "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6IjhZT1dKUUpONzFNazhWWU9uYzNlY3c9PSIsInZhbHVlIjoiTVFHTExqdmdXMk11SWlRcC90L24zb0VKbjdLYnlWSVBkekxYMUc0d20xcz0iLCJtYWMiOiJhZDkxZTg4NmUzODQzNjhkOThhMmNkYTg3ZDU4OGE0MWQ1NDVmMGIwMmM5ZGU2MjgxNjNiZDNmNTFkNjMyOGQ0IiwidGFnIjoiIn0="
AJAX_URL = "https://durg.ucanapply.com/get-result-details"

COLLEGE_FILE = "college-codes.txt"

# 2019 Roll Structure: 9 (Year) + CollegeCode + 009 (Course) + Serial
YEAR_CONST = "9"
COURSE_CONST = "008"

MAX_WORKERS = 60
REQUESTS_PER_SECOND = 100
SERIAL_START = 1
SERIAL_END = 300   # Adjusted to 300 for 2020

SAVE_FOLDER = "UG_BSC_FINAL_2020"
os.makedirs(SAVE_FOLDER, exist_ok=True)

# ==========================================
# LOAD COLLEGE CODES
# ==========================================

def load_college_codes():
    codes = []
    if not os.path.exists(COLLEGE_FILE):
        print(f"❌ Error: {COLLEGE_FILE} not found!")
        return []
    with open(COLLEGE_FILE, "r", encoding="utf-8") as f:
        lines = f.readlines()[1:]
        for line in lines:
            parts = line.strip().split("\t")
            if parts and parts[0].isdigit():
                codes.append(parts[0])
    return codes

# ==========================================
# TOKEN FETCH
# ==========================================

def get_session_tokens():
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": RESULT_PAGE_URL
    })

    try:
        r = session.get(RESULT_PAGE_URL, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")

        csrf_token = soup.find("input", {"name": "_token"})["value"]

        payload_template = {
            "COURSECD": soup.find("input", {"name": "COURSECD"})["value"],
            "SEMCODE": soup.find("input", {"name": "SEMCODE"})["value"],
            "RESULTTYPE": soup.find("input", {"name": "RESULTTYPE"})["value"],
            "session": soup.find("input", {"name": "session"})["value"],
            "tcc": soup.find("input", {"name": "tcc"})["value"],
            "p1": "",
            "all": ""
        }
        return session.cookies.get_dict(), payload_template, csrf_token
    except Exception as e:
        print(f"❌ Failed to fetch initial tokens: {e}")
        exit()

# ==========================================
# WORKER
# ==========================================

def fetch_roll(args):
    roll, template, token, cookies = args

    headers = {
        "X-Requested-With": "XMLHttpRequest",
        "X-CSRF-TOKEN": token,
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "User-Agent": "Mozilla/5.0"
    }

    data = template.copy()
    data["EXAMROLLNUMBER"] = roll
    data["_token"] = token

    try:
        res = requests.post(AJAX_URL, data=data, headers=headers, cookies=cookies, timeout=10)
        if res.status_code == 200:
            rj = res.json()
            if "html" in rj and len(rj["html"]) > 100:
                with open(os.path.join(SAVE_FOLDER, f"{roll}.html"), "w", encoding="utf-8") as f:
                    f.write(rj["html"])
                return "SAVED"
    except:
        pass

    return "FAILED"

# ==========================================
# MAIN ENGINE
# ==========================================

def main():
    print("🚀 UG BSC FINAL 2020 MULTI-COLLEGE SCRAPER")
    print(f"📁 Target Folder: {SAVE_FOLDER}")

    # Smart Resume logic
    existing_files = set(f.replace(".html", "") for f in os.listdir(SAVE_FOLDER) if f.endswith(".html"))
    if existing_files:
        print(f"🔎 Found {len(existing_files)} existing files. Skipping them.")

    colleges = load_college_codes()
    if not colleges: return

    cookies, template, token = get_session_tokens()
    total_newly_saved = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

        for college in colleges:
            print(f"\n🏫 Processing College: {college}")

            consecutive_failures = 0
            started = any(roll.startswith(f"{YEAR_CONST}{college}") for roll in existing_files)

            for batch_start in range(SERIAL_START, SERIAL_END + 1, REQUESTS_PER_SECOND):

                batch_end = min(batch_start + REQUESTS_PER_SECOND - 1, SERIAL_END)
                serial_range = range(batch_start, batch_end + 1)

                tasks = []
                for serial in serial_range:
                    # New Roll Format: 9 + 334 + 009 + 0068
                    roll = f"{YEAR_CONST}{college}{COURSE_CONST}{serial:04d}"

                    if roll in existing_files:
                        continue

                    tasks.append((roll, template, token, cookies))

                if not tasks:
                    continue

                start_time = time.time()
                results = list(executor.map(fetch_roll, tasks))

                batch_stop_triggered = False
                for status in results:
                    if status == "SAVED":
                        total_newly_saved += 1
                        consecutive_failures = 0
                        started = True
                        print(f"✅ New Saved: {total_newly_saved} | Current: {college}", end="\r")
                    else:
                        if started:
                            consecutive_failures += 1
                            if consecutive_failures >= 5: # Increased tolerance for 2019
                                print(f"\n🛑 Stopping College {college} (5 consecutive empty results)")
                                batch_stop_triggered = True
                                break

                if batch_stop_triggered:
                    break

                elapsed = time.time() - start_time
                if elapsed < 1:
                    time.sleep(1 - elapsed)

    print(f"\n\n🎉 Task Finished.")
    print(f"📊 Newly Downloaded: {total_newly_saved}")
    print(f"📂 Total records now in folder: {len(os.listdir(SAVE_FOLDER))}")

if __name__ == "__main__":
    main()

In [ ]:
# @title ba 2019 to 2023

import requests
from bs4 import BeautifulSoup
import os
import concurrent.futures
import time

# ==========================================
#        MASTER CONFIGURATION
# ==========================================
COLLEGE_CODE_FILE = 'college-codes.txt'
AJAX_URL = "https://durg.ucanapply.com/get-result-details"

MAX_SERIAL_PER_COLLEGE = 600
EARLY_STOP_THRESHOLD = 10
CONSECUTIVE_FAIL_LIMIT = 15
BASE_FOLDER = "BA_Final_Year_Results"

# ==========================================
#        BATCH DEFINITIONS (FULL URLs)
# ==========================================
BATCH_CONFIGS = [
    {
        "id": "B1_2019",
        "url": "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6IlVQSzhJbWNNRzBjWkEvRlhNd0lIaWc9PSIsInZhbHVlIjoiTnBGalZIOFgvZGdUN1oyZUc5ZFhPZ1Y3TUV3dVVpTTlWSDFVaWdXM0JPbz0iLCJtYWMiOiJjNzk2NjQyZTYwOTI4OThlNjljNTM3ZDZlZWYyMDJmMDExNThkOGU1MTliYTc4ZDJkNTI2YzA5N2RlZWI1MWQxIiwidGFnIjoiIn0=",
        "roll_algo": lambda coll, i: f"9{coll}003{i:04d}"
    },
    {
        "id": "B2_2020",
        "url": "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6InU5U1dGVk5UVDJYdVIwL1lhKzdNV3c9PSIsInZhbHVlIjoiS0VGRTFqaXh1MFhuWUltWDF0c0U5dUc5bmR3b2tRVEJ5YWNhTUpFazB4az0iLCJtYWMiOiJjOWRmMDRiZTEyMGVjY2UzZWVlNTcyNTJmZDJiMjMzNTYwNzhkOTVlMTlhZGZjNmRmNWU2NThhYzU3Y2Y5ODAyIiwidGFnIjoiIn0=",
        "roll_algo": lambda coll, i: f"9{coll}002{i:04d}"
    },
    {
        "id": "B3_2021",
        "url": "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6Imp5RHdBYVBnSzRoMDFtc2loaEZuWXc9PSIsInZhbHVlIjoiNDF0bjFZQ3lSNUJlbUZFam42ZVFrN3BLSXYwM2E1RE1NR0J2RDErcWw5UT0iLCJtYWMiOiI3YTIyN2Y5ZDA2YTBlNDcwNzkyYWI1ODlkOGNjOGRiMTM2Y2IyYzAxMGRiYTQ0Mjg5NzA4YjcyM2QzMjQ0OWMyIiwidGFnIjoiIn0=",
        "roll_algo": lambda coll, i: f"2{coll}002{i:04d}"
    },
    {
        "id": "B4_2022",
        "url": "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6IndSdlp1Nm9kWmJyWXQwcnhzUHN5aVE9PSIsInZhbHVlIjoiRW90Z2FxZG9ETjFMNEhHRzUxZEVhRFNrUW4za2dnUTcxd0hlTW5kRXhXMD0iLCJtYWMiOiI4TFphNGFiNTNjNTAzODJjZjFhZTQwOGQzOWYxZDBkNjQzYjllZmJjNzEyNzQxNmMxYTcwMTBlM2M5MDIwMDE1IiwidGFnIjoiIn0=",
        "roll_algo": lambda coll, i: f"2{coll}001{i:04d}"
    },
    {
        "id": "B5_2023",
        "url": "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6IlpCbHFkdzlYNGlGNFBqOGV2RDU2b0E9PSIsInZhbHVlIjoiOHV3SldzVndTcUFabktQMVYrdVhnZkJkdFFwMTcyNzd2S0FEMUx6MDNFUT0iLCJtYWMiOiJlOWJlYzUwYTVmN2Q0MmFkYzhhYjEyZTNhYzc5YjExOTg2MjdlODdhMTEzNTk0ODFiNTZiNDhlYWMyM2IyZWRhIiwidGFnIjoiIn0=",
        "roll_algo": lambda coll, i: f"3{coll}001{i:04d}"
    }
]

# ==========================================
#           CORE FUNCTIONS
# ==========================================

def load_college_codes():
    codes = []
    if not os.path.exists(COLLEGE_CODE_FILE):
        return []
    with open(COLLEGE_CODE_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if parts and parts[0].isdigit():
                codes.append(parts[0])
    return codes


def get_session_data(url):
    s = requests.Session()
    s.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
    })

    try:
        r = s.get(url, timeout=15)
        soup = BeautifulSoup(r.text, 'html.parser')

        token = soup.find('input', {'name': '_token'})['value']
        payload = {
            'COURSECD': soup.find('input', {'name': 'COURSECD'})['value'],
            'SEMCODE': soup.find('input', {'name': 'SEMCODE'})['value'],
            'RESULTTYPE': soup.find('input', {'name': 'RESULTTYPE'})['value'],
            'session': soup.find('input', {'name': 'session'})['value'],
            'tcc': soup.find('input', {'name': 'tcc'})['value'],
            'p1': '',
            'all': ''
        }

        return s.cookies.get_dict(), payload, token
    except:
        return None, None, None


def fetch_result(roll, url, payload, token, cookies):
    headers = {
        'X-Requested-With': 'XMLHttpRequest',
        'X-CSRF-TOKEN': token,
        'Referer': url,
        'Content-Type': 'application/x-www-form-urlencoded'
    }

    data = payload.copy()
    data['EXAMROLLNUMBER'] = roll
    data['_token'] = token

    try:
        r = requests.post(AJAX_URL, data=data, headers=headers, cookies=cookies, timeout=7)
        if r.status_code == 200:
            rj = r.json()
            if 'html' in rj and len(rj['html']) > 200:
                return roll, rj['html']
    except:
        pass

    return roll, None


def process_batch(config, colleges):
    print(f"\n🚀 STARTING BATCH: {config['id']}")

    cookies, payload, token = get_session_data(config['url'])
    if not token:
        print("❌ Failed to initialize session")
        return

    batch_folder = os.path.join(BASE_FOLDER, config['id'])
    os.makedirs(batch_folder, exist_ok=True)

    for coll in colleges:

        completed_flag = os.path.join(batch_folder, f"_completed_{coll}.txt")
        if os.path.exists(completed_flag):
            print(f"⏩ College {coll} already completed.")
            continue

        active_college = False
        consecutive_fails = 0
        total_saved = 0

        print(f"\n🏢 College {coll} processing...")

        for chunk_start in range(1, MAX_SERIAL_PER_COLLEGE + 1, 50):

            tasks = []

            for i in range(chunk_start, min(chunk_start + 50, MAX_SERIAL_PER_COLLEGE + 1)):
                roll = config['roll_algo'](coll, i)
                file_path = os.path.join(batch_folder, f"{roll}.html")

                if os.path.exists(file_path):
                    continue

                tasks.append(roll)

            if not tasks:
                continue

            with concurrent.futures.ThreadPoolExecutor(max_workers=30) as executor:
                futures = {
                    executor.submit(fetch_result, r, config['url'], payload, token, cookies): r
                    for r in tasks
                }

                results = []
                for future in concurrent.futures.as_completed(futures):
                    results.append(future.result())

                results.sort(key=lambda x: x[0])

                for roll, html in results:
                    serial_part = int(roll[-4:])
                    file_path = os.path.join(batch_folder, f"{roll}.html")

                    if html:
                        active_college = True
                        consecutive_fails = 0
                        total_saved += 1

                        with open(file_path, "w", encoding="utf-8") as f:
                            f.write(html)
                    else:
                        consecutive_fails += 1

                    if not active_college and serial_part >= EARLY_STOP_THRESHOLD:
                        break

                    if active_college and consecutive_fails >= CONSECUTIVE_FAIL_LIMIT:
                        break

                if (not active_college and serial_part >= EARLY_STOP_THRESHOLD) or \
                   (active_college and consecutive_fails >= CONSECUTIVE_FAIL_LIMIT):
                    break

        with open(completed_flag, "w") as f:
            f.write("DONE")

        status = "✅ Found" if active_college else "❌ No Data"
        print(f"🏢 College {coll} | {status} | Saved: {total_saved}")


# ==========================================
#           MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    colleges = load_college_codes()

    if not colleges:
        print("college-codes.txt not found!")
    else:
        for config in BATCH_CONFIGS:
            process_batch(config, colleges)
            time.sleep(5)

In [ ]:
# @title bcom

import requests
from bs4 import BeautifulSoup
import os
import concurrent.futures
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ==========================================
#        MASTER CONFIGURATION
# ==========================================
COLLEGE_CODE_FILE = 'college-codes.txt'
AJAX_URL = "https://durg.ucanapply.com/get-result-details"

MAX_WORKERS = 300              # Set to 300 for target speed
MAX_SERIAL_PER_COLLEGE = 9999  # Scan full 4-digit range
CONSECUTIVE_FAIL_LIMIT = 20    # Stop ONLY after finding students and hitting a gap
BASE_FOLDER = "BCOM_Final_Year_Results"

# ==========================================
#        BATCH DEFINITIONS FOR BCOM
# ==========================================
BATCH_CONFIGS = [
    {
        "id": "B1_2019",
        "url": "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6IlVIOWNjdnBjZ0pERHV5ekZ6L0ErcXc9PSIsInZhbHVlIjoiUDZwL2xSRG8ybkFuTUdneUZKUXh0UzVHTm93dCtLWjg3b2F6b0JzR1I2Yz0iLCJtYWMiOiJjM2IzMjE2ZDNiYWIzZTE1ZGM4MTE1NjJlYmExZjNiZjZhMTNiNDVhZWY4YmQxOGNhMDNhZjEyNWI5MzAwZWQ2IiwidGFnIjoiIn0=",
        "roll_algo": lambda coll, i: f"9{coll}006{i:04d}"
    },
    {
        "id": "B2_2020",
        "url": "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6IlNQbU5xbTlsL2RPamR2OXE3MTR3aVE9PSIsInZhbHVlIjoiNWZOU3ZGSDRGRTFybUV3dExVVkJQT3hVK0x1bWQ5amJ2eGZsUW5xLzhLTT0iLCJtYWMiOiIwYTQ5ZWY2YmFkYWI3Y2I3YzJhNWUxNjBjM2MzM2EyOTMzOWJiYTNiM2Q4NjhjNDAxMjRlYzEwMjMwNDhiNGRmIiwidGFnIjoiIn0=",
        "roll_algo": lambda coll, i: f"9{coll}005{i:04d}"
    },
    {
        "id": "B3_2021",
        "url": "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6Ik52UFZaTUw1bjdXLzR3VmsxN1QyYUE9PSIsInZhbHVlIjoiMzl4YlpVMHlzd25sMjBoM2xTL0xWZitWdTI4a3p0NmJTdTd2QWtISGRJWT0iLCJtYWMiOiIwY2Y1Y2MxYTBiMjljODA2MjE1NTZiMTI5NTEzN2E0ZTA3NzljNWE3ZmZlNDBiMGY2OWIxNGY0ZGQyOWUzZWM3IiwidGFnIjoiIn0=",
        "roll_algo": lambda coll, i: f"2{coll}005{i:04d}"
    },
    {
        "id": "B4_2022",
        "url": "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6Imdlc0kwZmVSTEt2REExU2xwY0orenc9PSIsInZhbHVlIjoiZ2ZjdVFleGVrYVEwbHFselBQUDNnYjBGZ2kvaTN2THJIZFlxMXRZM0FzMD0iLCJtYWMiOiIxNzUwZGJjMjI5NTlkY2JmODk4MzcxZTg4YTMxYWE3Nzc2NTE0NjViMjdjZWVlYmU3NmNhZDA4ZGQ1ZmJjMDkzIiwidGFnIjoiIn0=",
        "roll_algo": lambda coll, i: f"2{coll}004{i:04d}"
    },
    {
        "id": "B5_2023",
        "url": "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6IlpOaTdBc1M4dWpnUDErek5FUXQydlE9PSIsInZhbHVlIjoiNUVKakNpSGN0NXNRcS90QWdMZW5hcWFNSnZiWEVWNDNQempIRXBya2RFWT0iLCJtYWMiOiI2MDliY2E0OTk5OWMyYWFhN2IzNzMyMWFkYmY2YWVlNzM3MDIxMGYwOTRhMGYyNmYyMDc3NWIwNWU5ZjhjYjZjIiwidGFnIjoiIn0=",
        "roll_algo": lambda coll, i: f"22{coll}005{i:04d}"
    },
    {
        "id": "B6_2024",
        "url": "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6IkFQVG1CMmNoTDVUQXREWUJpQ1AxUnc9PSIsInZhbHVlIjoiZm9ZeGFWZXRYSFprUVVEQjF3MlVmM09OM3BoeGlUb0NJbWcrZXZ5aVdDTT0iLCJtYWMiOiJlMGU3MWQ2NWIzODQ4YWVhN2UyZmRmYzNiMGEzNjI5NDVkNmRjYmI1MjIxZWVlMDkwNDliNDUwNTE4NDdhYmY3IiwidGFnIjoiIn0=",
        "roll_algo": lambda coll, i: f"4{coll}{i:04d}"
    },
    {
        "id": "B7_2025",
        "url": "https://durg.ucanapply.com/result-details?tcc=eyJpdiI6ImNBcFVGQXpSdmhKOGJpVXpKWVRYNnc9PSIsInZhbHVlIjoidkNDeVRQYnFPL296ZHZBQUc0S1gzSnFoTktUd2daRFVaalp3eDBsSjNKOD0iLCJtYWMiOiI2NDUyMTU2ZTNiMzJhMjUyNjU4MmRlZjI3MTlhYTllNmRiNjczZmFmOGEzMzRhMjJiYmY3NDUxY2JiOGYzM2IzIiwidGFnIjoiIn0=",
        "roll_algo": lambda coll, i: f"5{coll}{i:04d}"
    }
]

# ==========================================
#           CORE FUNCTIONS
# ==========================================

def load_college_codes():
    codes = []
    if not os.path.exists(COLLEGE_CODE_FILE): return []
    with open(COLLEGE_CODE_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if parts[0].isdigit(): codes.append(parts[0])
    return codes

def get_high_speed_session(url):
    s = requests.Session()
    # Increase pool size to handle hundreds of concurrent connections
    adapter = HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
    s.mount("https://", adapter)
    s.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'})

    try:
        r = s.get(url, timeout=15)
        soup = BeautifulSoup(r.text, 'html.parser')
        token = soup.find('input', {'name': '_token'})['value']
        payload = {
            'COURSECD': soup.find('input', {'name': 'COURSECD'})['value'],
            'SEMCODE': soup.find('input', {'name': 'SEMCODE'})['value'],
            'RESULTTYPE': soup.find('input', {'name': 'RESULTTYPE'})['value'],
            'session': soup.find('input', {'name': 'session'})['value'],
            'tcc': soup.find('input', {'name': 'tcc'})['value'],
            'p1': '', 'all': ''
        }
        return s, payload, token
    except: return None, None, None

def fetch_result(session, roll, url, payload, token):
    headers = {
        'X-Requested-With': 'XMLHttpRequest',
        'X-CSRF-TOKEN': token,
        'Referer': url,
        'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8'
    }
    data = payload.copy()
    data['EXAMROLLNUMBER'] = roll
    data['_token'] = token
    try:
        # Using session.post is faster as it reuses the connection
        r = session.post(AJAX_URL, data=data, headers=headers, timeout=6)
        if r.status_code == 200:
            rj = r.json()
            if 'html' in rj and len(rj['html']) > 200:
                return roll, rj['html']
    except: pass
    return roll, None

def process_batch(config, colleges):
    print(f"\n🚀 INITIALIZING BATCH: {config['id']}")
    session, payload, token = get_high_speed_session(config['url'])
    if not token:
        print("❌ Failed to initialize session token")
        return

    batch_folder = os.path.join(BASE_FOLDER, config['id'])
    os.makedirs(batch_folder, exist_ok=True)

    for coll in colleges:
        found_any_in_college = False
        consecutive_fails = 0
        total_saved = 0

        # We process the college in blocks (one full thread dump)
        for chunk_start in range(1, MAX_SERIAL_PER_COLLEGE + 1, MAX_WORKERS):
            tasks = []
            for i in range(chunk_start, min(chunk_start + MAX_WORKERS, MAX_SERIAL_PER_COLLEGE + 1)):
                roll = config['roll_algo'](coll, i)

                # Check if file exists before queuing request (prevent redownloading)
                file_path = os.path.join(batch_folder, f"{roll}.html")
                if not os.path.exists(file_path):
                    tasks.append(roll)

            if not tasks:
                continue

            with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                future_to_roll = {executor.submit(fetch_result, session, r, config['url'], payload, token): r for r in tasks}

                results = []
                for future in concurrent.futures.as_completed(future_to_roll):
                    results.append(future.result())

                results.sort(key=lambda x: x[0])

                college_done = False

                for roll, html in results:
                    if html:
                        found_any_in_college = True
                        consecutive_fails = 0
                        total_saved += 1

                        file_path = os.path.join(batch_folder, f"{roll}.html")
                        with open(file_path, "w", encoding="utf-8") as f:
                            f.write(html)
                    else:
                        if found_any_in_college:
                            consecutive_fails += 1

                    # End detection: If we found someone and now hit a gap of 20, stop.
                    if found_any_in_college and consecutive_fails >= CONSECUTIVE_FAIL_LIMIT:
                        college_done = True
                        break

                print(f"\r🏢 College {coll} | Scanned to {chunk_start + MAX_WORKERS} | Found: {total_saved}", end="", flush=True)
                if college_done: break

        if total_saved > 0:
            print(f"\n✅ Finished College {coll}. Total Saved: {total_saved}")
        else:
            print(f"\r🏢 College {coll} | ❌ No students found in range 0001-9999.")

# ==========================================
#           MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    college_list = load_college_codes()
    if not college_list:
        print("college-codes.txt not found! Please create it with college codes.")
    else:
        for batch_cfg in BATCH_CONFIGS:
            process_batch(batch_cfg, college_list)
            time.sleep(2) # Give a brief pause between batches